# AI-MeshOptimizer -- Training Notebook

Trains the MeshGNN edge-importance model used by **Feature-Weighted QEM Simplification**
(an open-source, learned alternative to ZRemesher).

```
High Poly Mesh -> Feature Extraction -> GNN -> Edge Importance -> Feature-Weighted QEM -> Low Poly Mesh
```

Fully autonomous: run every cell top to bottom (Runtime -> Run all) and it
clones the public repo, generates a procedural training dataset across four
categories, preprocesses it, trains, plots, and runs inference -- no manual
edits required. Designed for **Google Colab Free (T4 GPU)**, no paid services.

Dataset categories (see `dataset/make_sample_dataset.py`):

| category | meaning |
|---|---|
| `complex_knot` | complex: twisting torus-knot topology, high curvature |
| `organic_blob` | complex: noise-displaced "sculpt", organic surface |
| `box_simple` | simple: flat box, few polygons |
| `sphere_simple` | simple: low-poly sphere, few polygons |


## 1. Check GPU

In [ ]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Colab -> Runtime > Change runtime type > T4 GPU.")


## 2. Install dependencies

Colab already ships a CUDA build of PyTorch, so we only add what this project
needs on top of it.

In [ ]:
!pip install -q torch-geometric
!pip install -q trimesh open3d pyfqmr networkx rtree tqdm matplotlib plyfile

import torch, torch_geometric
print("torch:", torch.__version__)
print("torch_geometric:", torch_geometric.__version__)


## 3. Clone the repository

Public repo, no setup needed: https://github.com/Gentleman12-sys/remesher_ai

The project root is auto-detected inside the clone (works whether the repo
puts `inference/remesh.py` at the top level or inside a subfolder).

In [ ]:
import os

REPO_URL = "https://github.com/Gentleman12-sys/remesher_ai.git"
CLONE_DIR = "/content/remesher_ai_repo"

if not os.path.exists(CLONE_DIR):
    !git clone "{REPO_URL}" "{CLONE_DIR}"
else:
    print("Already cloned at", CLONE_DIR)
    !git -C "{CLONE_DIR}" pull

def _find_project_root(base):
    for root, dirs, files in os.walk(base):
        if os.path.basename(root) == "inference" and "remesh.py" in files:
            return os.path.dirname(root)
    return None

PROJECT_DIR = _find_project_root(CLONE_DIR)
assert PROJECT_DIR is not None, (
    f"Could not find inference/remesh.py anywhere under {CLONE_DIR} -- "
    "check that https://github.com/Gentleman12-sys/remesher_ai actually contains "
    "the AI-MeshOptimizer project."
)

%cd {PROJECT_DIR}
import sys
sys.path.append(PROJECT_DIR)
print("Project root:", os.getcwd())


## 4. Configuration

One switch controls the whole run:

- `QUICK_TEST = True` -- a fast end-to-end confirmation pass (~a few minutes):
  small dataset, few epochs. Good for your first run, to make sure everything
  works before committing a Colab session to the real thing.
- `QUICK_TEST = False` -- the real training run: **750 meshes per category**
  (3,000 total -- within the requested 500-1000/category range) and 100
  epochs, matching the project spec (`AdamW`, `CosineAnnealingLR`,
  `batch_size=8`, `lr=0.0001`).

Rough budget for `QUICK_TEST = False` on a free Colab T4 (varies by GPU load):
dataset generation + preprocessing ~30-45 min, disk ~3-4 GB, training
~1.5-3 hours for 100 epochs. Lower `EPOCHS` below if your session has less
time -- everything downstream just uses whatever `EPOCHS`/`COUNT_PER_CATEGORY`
you set here.

In [ ]:
QUICK_TEST = True  # <-- flip to False for the real 750/category, 100-epoch run

if QUICK_TEST:
    COUNT_PER_CATEGORY = 20
    EPOCHS = 15
else:
    COUNT_PER_CATEGORY = 750
    EPOCHS = 100

BATCH_SIZE = 8
LEARNING_RATE = 0.0001

# per-category QEM reduction target: complex meshes need aggressive
# reduction, simple ones need a much gentler touch (see README.md > Dataset)
CATEGORY_CONFIG = {
    "complex_knot":  {"reduction_ratio": 0.08, "min_faces": 200},
    "organic_blob":  {"reduction_ratio": 0.08, "min_faces": 200},
    "box_simple":    {"reduction_ratio": 0.40, "min_faces": 20},
    "sphere_simple": {"reduction_ratio": 0.40, "min_faces": 20},
}

print(f"QUICK_TEST={QUICK_TEST}")
print(f"COUNT_PER_CATEGORY={COUNT_PER_CATEGORY} ({COUNT_PER_CATEGORY * len(CATEGORY_CONFIG)} meshes total)")
print(f"EPOCHS={EPOCHS} BATCH_SIZE={BATCH_SIZE} LEARNING_RATE={LEARNING_RATE}")


## 5. Generate the raw dataset

Procedurally generates `COUNT_PER_CATEGORY` meshes per category into
`dataset/raw/<category>/`. Pure trimesh/numpy, no external download --
typically well under a minute even at 750/category (mesh *creation* is cheap;
the QEM simplification in the next step is the slower part).

In [ ]:
!python dataset/make_sample_dataset.py \
    --count_per_category {COUNT_PER_CATEGORY} \
    --out_dir dataset/raw \
    --seed 0


## 6. Build training graphs (preprocessing)

Runs `preprocessing/generate_pairs.py` once per category with its own
`--reduction_ratio`/`--min_faces` from the config above: extracts node/edge
features, computes QEM-based edge-importance labels, generates a low-poly
target mesh for the Chamfer/normal loss terms, and saves one PyG graph per
mesh to `dataset/processed/`. This is the slower step (roughly ~0.5-1s per
complex mesh, near-instant per simple mesh).

In [ ]:
for category, cfg in CATEGORY_CONFIG.items():
    print(f"\n=== {category} (reduction_ratio={cfg['reduction_ratio']}, min_faces={cfg['min_faces']}) ===")
    !python preprocessing/generate_pairs.py \
        --input_dir dataset/raw/{category} \
        --reduction_ratio {cfg['reduction_ratio']} \
        --min_faces {cfg['min_faces']}


## 7. Inspect the training dataset

In [ ]:
from training.dataset import MeshPairDataset
from collections import Counter

dataset = MeshPairDataset("dataset/processed")
print(f"{len(dataset)} graphs total")

counts = Counter()
for p in dataset.paths:
    # filenames are "<category>_####.pt"
    counts["_".join(p.stem.split("_")[:-1])] += 1
for category, n in counts.items():
    print(f"  {category:16s} {n}")

g = dataset[0]
print("\nexample graph:", g)
print("node features:", g.x.shape, "edge_index:", g.edge_index.shape, "edge_attr:", g.edge_attr.shape)


## 8. Train the model

`AdamW` + `CosineAnnealingLR`, matching the project spec.

In [ ]:
!python training/train.py \
    --processed_dir dataset/processed \
    --checkpoint_dir checkpoints \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LEARNING_RATE} \
    --val_split 0.15 \
    --device auto


## 9. Plot training curves

`train.py` already saved `checkpoints/training_curves.png` (loss, accuracy, edge-prediction precision/recall/F1) -- display it here.

In [ ]:
from IPython.display import Image, display
display(Image(filename="checkpoints/training_curves.png"))


## 10. Run inference on a sample mesh

Uses `checkpoints/best_model.pt` to predict edge importance, then runs
Feature-Weighted QEM simplification down to `--target_faces`.

In [ ]:
import glob

complex_samples = sorted(glob.glob("dataset/raw/complex_knot/*.obj"))
sample_mesh = complex_samples[0]
print("Using sample mesh:", sample_mesh)

!python inference/remesh.py {sample_mesh} /content/output_low.obj \
    --target_faces 500 \
    --checkpoint checkpoints/best_model.pt \
    --compare


## 11. Visualize before / after (matplotlib)

In [ ]:
import trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def plot_mesh(ax, mesh, title):
    tris = mesh.vertices[mesh.faces]
    coll = Poly3DCollection(tris, alpha=0.85, edgecolor="k", linewidths=0.15)
    coll.set_facecolor((0.4, 0.6, 0.9))
    ax.add_collection3d(coll)
    bounds = mesh.vertices
    ax.set_xlim(bounds[:, 0].min(), bounds[:, 0].max())
    ax.set_ylim(bounds[:, 1].min(), bounds[:, 1].max())
    ax.set_zlim(bounds[:, 2].min(), bounds[:, 2].max())
    ax.set_title(f"{title}\n{len(mesh.faces)} faces")

original = trimesh.load(sample_mesh, process=False, force="mesh")
simplified = trimesh.load("/content/output_low.obj", process=False, force="mesh")

fig = plt.figure(figsize=(12, 6))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122, projection="3d")
plot_mesh(ax1, original, "Original (High Poly)")
plot_mesh(ax2, simplified, "AI-MeshOptimizer (Low Poly)")
plt.tight_layout()
plt.show()


## Notes

- `checkpoints/best_model.pt` is the artifact `inference/remesh.py` needs.
- To scale up further, just raise `COUNT_PER_CATEGORY` in step 4 and re-run
  from step 5 -- everything downstream is unchanged.
- To train on real-world data instead of/in addition to the procedural set,
  drop meshes into `dataset/raw/<your_category>/` and add a matching entry to
  `CATEGORY_CONFIG`, or just point `preprocessing/generate_pairs.py` at them
  directly (see README.md > Dataset).
- pyfqmr is only used by `generate_pairs.py`/`make_sample_dataset.py`'s low-poly
  targets for supervision; at inference time, edge collapse is driven by
  `utils/qem.py`'s own weighted simplifier so the GNN's per-edge importance can
  actually bias which edges survive.